# BoTorch Acquisition Functions

This tutorial shows how to use the BoTorch acquisition function wrappers from
[`alf_tools.optimizer.acquisition_functions.botorch_acquisition_function`](https://instadeepai.github.io/alf/api/alf_tools/optimizer/acquisition_functions/botorch_acquisition_function/).

These wrappers accept **either** a native BoTorch `Model` **or** an ALF `BaseModel`
and insert a `BoTorchModelAdapter` automatically when needed. Here we use ALF's
`GPModel` as the surrogate to demonstrate the full integration.

`BotorchAcquisitionFunction` is configured via a `BotorchAcquisitionConfig` that
selects a BoTorch acquisition class by name from `ACQUISITION_REGISTRY`. It
implements the ALF `AcquisitionFunction` interface: `__call__(candidates, state)
→ LabelledCandidates`.

### Environment Setup

From the `/tutorials` directory:
```
uv sync
source .venv/bin/activate
```
Select `.venv` as the kernel when prompted.

## 1. Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from alf_core.dataclasses.candidate import Candidate, Modality
from alf_core.dataclasses.labelled_candidates import LabelledCandidates
from alf_tools.models import FeaturizerConfig, GPModel, GPModelConfig, GPTrainConfig
from alf_tools.optimizer.acquisition_functions.botorch_acquisition_function import (
    ACQUISITION_REGISTRY,
    BotorchAcquisitionConfig,
    BotorchAcquisitionFunction,
)
from alf_tools.utils.botorch_utils import candidates_to_tensor

print("Imports successful")
print(f"Registered acquisition functions: {sorted(ACQUISITION_REGISTRY)}")

## 2. Train a GP Surrogate

We fit a `GPModel` to a synthetic 1D function.  The `"precomputed"` featuriser
accepts `Candidate.data` values that are already `np.ndarray` or `torch.Tensor`,
so no custom featuriser function is needed for tabular data.

In [ ]:
np.random.seed(42)
X_MIN, X_MAX = 0.0, 2 * np.pi
N_TRAIN = 20


def true_fn(x):
    return np.sin(x) + 0.5 * np.sin(3 * x)


x_train = np.sort(np.random.uniform(X_MIN, X_MAX, N_TRAIN))
y_train = true_fn(x_train) + np.random.randn(N_TRAIN) * 0.1


def make_candidates(x_array):
    """Wrap 1D points as ALF Candidates with precomputed numpy features."""
    return [Candidate(data=np.array([xi]), modality=Modality.TABULAR) for xi in x_array]


train_data = LabelledCandidates(make_candidates(x_train), y_train)
search_candidates = make_candidates(np.linspace(X_MIN, X_MAX, 100))

gp = GPModel(
    model_config=GPModelConfig(kernel_type="matern", matern_nu=2.5),
    train_config=GPTrainConfig(num_iterations=100, log_frequency=100),
    featurizer_config=FeaturizerConfig(featurizer_type="precomputed"),
)
gp.train(train_data)

best_f = float(y_train.max())
print(f"GP trained on {N_TRAIN} points")
print(f"Best observed value (best_f): {best_f:.3f}")
print(f"Learned hyperparameters: {gp.get_hyperparameters()}")

## 3. `BotorchAcquisitionFunction`

`BotorchAcquisitionFunction` implements the ALF `AcquisitionFunction` interface:
`__call__(candidates, state) → LabelledCandidates`.  Configure it with a
`BotorchAcquisitionConfig` specifying the acquisition `name` (from
`ACQUISITION_REGISTRY`) and any required `kwargs` — `model` is always injected
from `state.surrogate.model` at call time and must **not** appear in `kwargs`.

> **In a real `DesignTask`**, `state` is the `State` object produced by
> `task.setup()`. Below we use a minimal stub to keep the example self-contained.

In [ ]:
# Minimal stub that mirrors state.surrogate.model used in a real DesignTask
class _MockSurrogate:
    def __init__(self, model):
        self.model = model


class _MockState:
    def __init__(self, model):
        self.surrogate = _MockSurrogate(model)


state = _MockState(gp)

# -- Expected Improvement --
ei_fn = BotorchAcquisitionFunction(
    BotorchAcquisitionConfig(name="expected_improvement", kwargs={"best_f": best_f})
)
ei_result = ei_fn(search_candidates, state)

# -- Upper Confidence Bound --
ucb_fn = BotorchAcquisitionFunction(
    BotorchAcquisitionConfig(name="upper_confidence_bound", kwargs={"beta": 2.0})
)
ucb_result = ucb_fn(search_candidates, state)

# -- Probability of Improvement --
poi_fn = BotorchAcquisitionFunction(
    BotorchAcquisitionConfig(name="probability_of_improvement", kwargs={"best_f": best_f})
)
poi_result = poi_fn(search_candidates, state)

# -- Log q-Noisy Expected Improvement (baseline = training points) --
X_baseline = candidates_to_tensor(train_data.candidates)  # (n_train, 1)
lnei_fn = BotorchAcquisitionFunction(
    BotorchAcquisitionConfig(
        name="log_noisy_expected_improvement", kwargs={"X_baseline": X_baseline}
    )
)
lnei_result = lnei_fn(search_candidates, state)

print(f"EI   scores: [{ei_result.labels.min():.4f}, {ei_result.labels.max():.4f}]")
print(f"UCB  scores: [{ucb_result.labels.min():.4f}, {ucb_result.labels.max():.4f}]")
print(f"POI  scores: [{poi_result.labels.min():.4f}, {poi_result.labels.max():.4f}]")
print(f"LNEI scores: [{lnei_result.labels.min():.4f}, {lnei_result.labels.max():.4f}]")
print(f"\nAll return LabelledCandidates: {type(ei_result).__name__}")

Switching acquisition functions is a one-line config change — no code changes
elsewhere.  The `log_noisy_expected_improvement` variant takes an `X_baseline`
tensor of already-observed points and is more robust to observation noise than
standard EI.

EI is non-negative and peaks near unexplored regions where improvement over
`best_f` is likely.  UCB (`mean + β·std`) balances exploitation (high mean) and
exploration (high uncertainty) with `β` controlling the trade-off.

In [ ]:
preds = gp.predict(search_candidates)
x_search = np.linspace(X_MIN, X_MAX, 100)

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

# Top: GP prediction
ax = axes[0]
ax.plot(x_search, true_fn(x_search), "k--", linewidth=1.2, alpha=0.6, label="True fn")
ax.scatter(x_train, y_train, s=50, color="black", zorder=5, label="Training data")
ax.plot(x_search, preds.means, color="#7b2d8b", linewidth=2, label="GP mean")
ax.fill_between(
    x_search,
    preds.means - 2 * np.sqrt(preds.variances),
    preds.means + 2 * np.sqrt(preds.variances),
    alpha=0.2,
    color="#7b2d8b",
    label="\u00b12\u03c3",
)
ax.axhline(best_f, color="green", linestyle=":", linewidth=1.5, label=f"best_f = {best_f:.2f}")
ax.set_ylabel("y")
ax.set_title("GP Surrogate (Mat\u00e9rn-2.5)")
ax.legend(fontsize=8, ncol=3)
ax.grid(True, alpha=0.3)

# Middle: EI
ax = axes[1]
ax.plot(x_search, ei_result.labels, color="#e74c3c", linewidth=2)
ax.fill_between(x_search, 0, ei_result.labels, alpha=0.2, color="#e74c3c")
ax.set_ylabel("EI score")
ax.set_title("Expected Improvement (non-negative; peaks where improvement is likely)")
ax.grid(True, alpha=0.3)

# Bottom: UCB
ax = axes[2]
ax.plot(x_search, ucb_result.labels, color="#2980b9", linewidth=2)
ax.fill_between(x_search, ucb_result.labels.min(), ucb_result.labels, alpha=0.2, color="#2980b9")
ax.set_ylabel("UCB score")
ax.set_xlabel("x")
ax.set_title("Upper Confidence Bound (\u03b2=2.0; balances mean and uncertainty)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Key Points

- **Single interface**: `BotorchAcquisitionFunction(BotorchAcquisitionConfig(name=...,
  kwargs=...))` wraps all supported BoTorch acquisition classes via `ACQUISITION_REGISTRY`.
- **Model-agnostic**: Accepts either a native BoTorch `Model` or an ALF `BaseModel` —
  `BoTorchModelAdapter` is inserted automatically.
- **ALF-compatible**: Implements `AcquisitionFunction` (`__call__(candidates, state) →
  LabelledCandidates`); works wherever ALF expects an `AcquisitionFunction`.
- **Config-driven**: Switching acquisition functions is a one-line config change.
- **Registry**: Available names: `"expected_improvement"`, `"upper_confidence_bound"`,
  `"probability_of_improvement"`, `"log_noisy_expected_improvement"`.
- **Variance required**: Models must provide variance estimates. Deterministic models
  (e.g. `CNNModel`) cannot be used with analytic BoTorch acquisitions and will raise a
  `ValueError` with a descriptive message.

See `tools/alf_tools/optimizer/acquisition_functions/botorch_acquisition_function.py` for the full
implementation and `tools/tests/optimizer/test_botorch_acqs.py` for more usage examples.